# 03. A2A (Agent2Agent) with LangSmith / Agent Server

> **주제**: 분산된 에이전트끼리 *서로 메시지를 주고받기* + 분산 트레이싱
>
> **원문**: https://docs.langchain.com/langsmith/server-a2a

---

## 이 노트북에서 배우는 것
1. A2A 프로토콜과 엔드포인트(`/a2a/{assistant_id}`), 지원 RPC 메서드
2. A2A 호환 에이전트의 **필수 조건**(state 에 `messages` 키)
3. `contextId` / `taskId` 로 대화 스레드 이어가기
4. 에이전트 ↔ 에이전트 대화 시뮬레이션
5. `contextId → thread_id` 매핑을 통한 LangSmith 분산 트레이싱

## 1. A2A 한눈에 보기

**A2A(Agent2Agent)** 는 *서로 다른 곳에 배포된 에이전트*들이 표준 메시징으로 협업하게 하는 프로토콜입니다. LangSmith 의 Agent Server 가 A2A 를 기본 지원하며, 대화 전반의 **분산 트레이싱**까지 유지합니다.

- **세 프로토콜 비교**: MCP(에이전트↔도구), ACP(에이전트↔에디터), **A2A(에이전트↔에이전트)**

**엔드포인트**: `POST /a2a/{assistant_id}` — JSON-RPC 2.0

**지원 RPC 메서드**
| 메서드 | 설명 |
|--------|------|
| `message/send` | 메시지 전송 후 완성된 응답 수신 |
| `message/stream` | SSE 로 실시간 스트리밍 |
| `tasks/get` | 작업 상태 확인 / 이전 결과 조회 |

**Agent Card 디스커버리**: `GET /.well-known/agent-card.json?assistant_id={assistant_id}` → 이름·설명·skills·I/O 모드·A2A 엔드포인트 반환

In [87]:
# A2A 는 langgraph-api 0.4.21 이상이 필요 (uv)
!uv pip install -q "langgraph-api>=0.4.21"
# 클라이언트 예제용
!uv pip install -q aiohttp openai

from dotenv import load_dotenv
load_dotenv("/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/.env", override=True)

True

## 2. A2A 호환 에이전트의 필수 조건

> **핵심**: 에이전트 state 에 **`messages` 키**가 있어야 A2A 의 text part 를 처리할 수 있습니다.

아래는 가장 단순한 state 정의입니다. `Context` 는 configurable 파라미터용, `State` 는 대화 상태용입니다.

In [88]:
from typing import Any, Dict, List, TypedDict
from dataclasses import dataclass

class Context(TypedDict):
    my_configurable_param: str

@dataclass
class State:
    messages: List[Dict[str, Any]]   # <- A2A text part 처리를 위해 필수

## 3. 완전한 에이전트 구현 (graph.py)

LangGraph 그래프 하나를 정의합니다. `call_model` 노드가 마지막 사용자 메시지를 받아 OpenAI 로 응답을 만들고, 누적된 `messages` 에 덧붙입니다.

이 파일을 `langgraph.json` 에 등록하면 Agent Server 가 자동으로 `/a2a/{assistant_id}` 엔드포인트를 노출합니다.

In [89]:
graph_py = '''
from __future__ import annotations
import os
from dataclasses import dataclass
from typing import Any, Dict, List, TypedDict

from langgraph.graph import StateGraph
from langgraph.runtime import Runtime
from openai import AsyncOpenAI

class Context(TypedDict):
    my_configurable_param: str

@dataclass
class State:
    messages: List[Dict[str, Any]]

async def call_model(state: State, runtime: Runtime[Context]) -> Dict[str, Any]:
    client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    latest_message = state.messages[-1] if state.messages else {}
    user_content = latest_message.get("content", "No message content")

    openai_messages = [
        {"role": "system",
         "content": "You are a helpful conversational agent. Keep responses brief and engaging."},
        {"role": "user", "content": user_content},
    ]

    try:
        response = await client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=openai_messages,
            max_tokens=100,
            temperature=0.7,
        )
        ai_response = response.choices[0].message.content
    except Exception as e:
        ai_response = f"I received your message but had trouble processing it. Error: {str(e)[:50]}..."

    response_message = {"role": "assistant", "content": ai_response}
    return {"messages": state.messages + [response_message]}

graph = (
    StateGraph(State, context_schema=Context)
    .add_node(call_model)
    .add_edge("__start__", "call_model")
    .compile()
)
'''

with open("graph.py", "w") as f:
    f.write(graph_py)
print("graph.py 생성 — langgraph.json 에 등록 후 `langgraph dev` 로 서버 실행")

graph.py 생성 — langgraph.json 에 등록 후 `langgraph dev` 로 서버 실행


## 4. 대화 이어가기: `contextId` 와 `taskId`

A2A 는 두 개의 식별자로 대화를 관리합니다.

- **`contextId`**: 메시지들을 하나의 **대화 스레드**로 묶음 (세션 ID 에 해당)
- **`taskId`**: 대화 안의 **개별 요청**을 식별

> **규칙**: 첫 메시지에는 둘 다 **생략** → 서버가 생성해서 돌려줌. 이후 메시지엔 둘 다 **포함**해서 스레드를 이어감.

## 5. 에이전트 ↔ 에이전트 대화 시뮬레이션

> ⚠️ **이 셀을 실행하기 전에 두 LangGraph dev 서버를 먼저 띄우세요.**
> 안 띄우면 `ClientConnectorError: Cannot connect to host 127.0.0.1:2024` 가 납니다.

§3 에서 만든 `graph.py` 와 `scripts/a2a/langgraph.json` 을 재사용해 **같은 그래프를 서로 다른 포트에 두 번 띄워** 에이전트 A·B 로 사용합니다.

### 준비 1) 터미널 두 개에서 서버 기동

LangGraph CLI 가 없다면 한 번만 설치:

```bash
source .venv/bin/activate
uv pip install -q "langgraph-cli[inmem]" "langgraph-api>=0.4.21"
```

**터미널 1 — Agent A (기본 포트 2024)**

```bash
source .venv/bin/activate
cd scripts/a2a
langgraph dev                       # -> http://127.0.0.1:2024
```

**터미널 2 — Agent B (포트 2025)**

```bash
source .venv/bin/activate
cd scripts/a2a
langgraph dev --port 2025           # -> http://127.0.0.1:2025
```

각 콘솔에 `Server is running at http://127.0.0.1:...` 가 보이면 준비 완료. 두 서버 모두 `langgraph.json` 의 `"graphs": {"agent": ...}` 를 노출합니다.

### 준비 2) `.env` 채우기 — `assistant_id` 는 **UUID** 여야 함

> ⚠️ **그래프 이름(`"agent"`) 을 그대로 넣으면 안 됩니다.** `langgraph-api` 는 `/assistants/{id}` 와 `/a2a/{id}` 의 path 파라미터를 **UUID 로 검증**해서, `"agent"` 같은 비-UUID 문자열을 받으면 `422 Unprocessable Entity` 로 거부합니다.

**왜 UUID 인가?**

한 서버에 같은 그래프를 **여러 변형(config·context 다름)** 으로 띄울 수 있고, 각 변형이 별개의 *assistant* 입니다. 그래서 식별자는 그래프 이름이 아니라 **assistant 단위 UUID** 가 됩니다. langgraph-api 는 그래프 하나당 "default assistant" 를 자동 생성하는데, 그 UUID 는 **그래프 이름으로부터 결정적(deterministic)** 으로 만들어집니다 — `uuid5(NAMESPACE_GRAPH, graph_id)`. 같은 그래프 이름이면 어느 서버에서나 같은 UUID 가 나옵니다.

**그래프 `agent` 의 UUID 구하는 법** (세 가지 중 아무거나):

방법 A — 서버에 직접 물어보기 (가장 확실, 한 줄):

```bash
curl -s -X POST http://127.0.0.1:2024/assistants/search \
  -H "Content-Type: application/json" -d '{"limit":10}'
```
응답 배열의 `assistant_id` 값이 그것입니다.

방법 B — 결정적 계산 (서버 없이도 동작):

```python
import uuid
NAMESPACE_GRAPH = uuid.UUID("6ba7b821-9dad-11d1-80b4-00c04fd430c8")
assistant_id = str(uuid.uuid5(NAMESPACE_GRAPH, "agent"))
# -> fe096781-5601-53d2-b2f6-0d3403f7e9ca
```

방법 C — `langgraph_sdk`:

```python
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")
print((await client.assistants.search())[0]["assistant_id"])
```

얻은 UUID 를 `.env` 에 넣습니다 (두 서버가 같은 그래프 `agent` 를 띄우므로 동일 UUID):

```
AGENT_A_ID=fe096781-5601-53d2-b2f6-0d3403f7e9ca
AGENT_B_ID=fe096781-5601-53d2-b2f6-0d3403f7e9ca
OPENAI_API_KEY=sk-...   # graph.py 의 call_model 이 OpenAI 호출
```

`.env` 수정 후 **노트북 상단의 `load_dotenv` 셀을 다시 실행하거나 커널 재시작** 해야 변경이 반영됩니다.

### 준비 3) 빠른 확인 (선택)

서버가 그 UUID 로 응답하는지 한 줄 점검 (UUID 는 본인 값으로):

```bash
UUID=fe096781-5601-53d2-b2f6-0d3403f7e9ca
curl -s "http://127.0.0.1:2024/.well-known/agent-card.json?assistant_id=$UUID" | head -c 200
curl -s "http://127.0.0.1:2025/.well-known/agent-card.json?assistant_id=$UUID" | head -c 200
```

JSON 일부가 보이면 OK. `422` 면 UUID 가 틀린 것, `Connection refused` 면 그 포트의 서버가 안 뜬 것.

---

준비가 됐으면 아래 셀에서 두 에이전트를 번갈아 호출해 한 에이전트의 응답을 다른 에이전트의 입력으로 넘기는 **핑퐁 대화**를 실행합니다.

`extract_text()` 는 A2A 응답에서 텍스트를 꺼내는 헬퍼입니다 — 응답 텍스트는 `artifacts[].parts[]` 또는 `status.message.parts[]` 에 들어 있습니다.

In [78]:
#!/usr/bin/env python3
import asyncio
import aiohttp
import os
import uuid

def extract_text(result: dict) -> str:
    for art in result.get("result", {}).get("artifacts", []) or []:
        for part in art.get("parts", []) or []:
            if part.get("kind") == "text" and part.get("text"):
                return part["text"]

    msg = (result.get("result", {}).get("status", {}) or {}).get("message", {}) or {}
    for part in msg.get("parts", []) or []:
        if part.get("kind") == "text" and part.get("text"):
            return part["text"]

    return "(no text found)"

In [79]:
async def send_message(session, port, assistant_id, text,
                       context_id=None, task_id=None, thread_id=None):
    """A2A message/send 한 번. thread_id 를 넘기면 JSON-RPC 최상위
    metadata.thread_id 로 실어보내, 여러 에이전트의 트레이스가 같은
    LangSmith thread 로 묶이게 한다."""
    url = f"http://127.0.0.1:{port}/a2a/{assistant_id}"

    message = {
        "role": "user",
        "parts": [{"kind": "text", "text": text}],
        "messageId": str(uuid.uuid4()),
    }
    if context_id:
        message["contextId"] = context_id
    if task_id:
        message["taskId"] = task_id

    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {"message": message},
    }
    if thread_id:
        # 분산 트레이싱 — 같은 thread_id 를 모든 에이전트가 공유하면
        # LangSmith 에서 하나의 thread 로 묶여 보인다.
        payload["metadata"] = {"thread_id": thread_id}

    headers = {"Accept": "application/json"}
    async with session.post(url, json=payload, headers=headers) as response:
        result = await response.json()

    returned_context_id = result.get("result", {}).get("contextId") or context_id
    returned_task_id = result.get("result", {}).get("id")
    return extract_text(result), returned_context_id, returned_task_id

In [80]:
async def simulate_conversation():
    agent_a_id = os.getenv("AGENT_A_ID")
    agent_b_id = os.getenv("AGENT_B_ID")

    if not agent_a_id or not agent_b_id:
        print("⚠️ .env 의 AGENT_A_ID / AGENT_B_ID 가 비어 있습니다.")
        print("   채우고 노트북 상단의 load_dotenv 셀을 다시 실행하거나 커널을 재시작하세요.")
        return

    seed = "Hello! Let's chat about your favorite hobby."

    print(f"💬 Seed (사용자 → Agent A):  {seed}")
    print(f"   🔵 Agent A  port 2024  agent_id={agent_a_id}")
    print(f"   🔴 Agent B  port 2025  agent_id={agent_b_id}")
    print("=" * 66, flush=True)

    # contextId · taskId 는 서버 단위 식별자 → 에이전트별로 따로 유지
    ctx_a, task_a = None, None
    ctx_b, task_b = None, None
    message = seed   # 라운드 시작 메시지: 첫 라운드는 seed, 이후는 B 의 답

    async with aiohttp.ClientSession() as session:
        for i in range(1, 4):
            print(f"\n── Round {i} " + "─" * 54, flush=True)

            # (1) Agent A 가 듣고 답한다 — A 자체 thread 사용
            print(f"  → Agent A 가 받음:  {message!r}", flush=True)
            reply_a, ctx_a, task_a = await send_message(
                session, 2024, agent_a_id, message,
                context_id=ctx_a, task_id=task_a,
            )
            print(f"  🔵 Agent A 답:     {reply_a}", flush=True)
            if reply_a == "(no text found)":
                print("     ⚠️ 응답에서 텍스트를 못 찾음 — 서버 로그를 확인하세요.", flush=True)

            # (2) Agent B 가 A 의 답을 듣고 답한다 — B 자체 thread 사용
            print(f"  → Agent B 가 받음:  {reply_a!r}", flush=True)
            reply_b, ctx_b, task_b = await send_message(
                session, 2025, agent_b_id, reply_a,
                context_id=ctx_b, task_id=task_b,
            )
            print(f"  🔴 Agent B 답:     {reply_b}", flush=True)
            if reply_b == "(no text found)":
                print("     ⚠️ 응답에서 텍스트를 못 찾음 — 서버 로그를 확인하세요.", flush=True)

            # 다음 라운드의 출발 메시지 = B 의 답 (A 가 그걸 듣고 다음 응답)
            message = reply_b

    print("\n" + "=" * 66)
    print(f"✅ 대화 종료 — 각 에이전트가 자기 thread 를 유지함")
    print(f"   Agent A thread contextId = {ctx_a}")
    print(f"   Agent B thread contextId = {ctx_b}", flush=True)


# 주피터에서는 await 로 실행 (스크립트면 asyncio.run(simulate_conversation()))
await simulate_conversation()

💬 Seed (사용자 → Agent A):  Hello! Let's chat about your favorite hobby.
   🔵 Agent A  port 2024  agent_id=fe096781-5601-53d2-b2f6-0d3403f7e9ca
   🔴 Agent B  port 2025  agent_id=fe096781-5601-53d2-b2f6-0d3403f7e9ca

── Round 1 ──────────────────────────────────────────────────────
  → Agent A 가 받음:  "Hello! Let's chat about your favorite hobby."
  🔵 Agent A 답:     Hello! I'm here to chat. What's your favorite hobby?
  → Agent B 가 받음:  "Hello! I'm here to chat. What's your favorite hobby?"
  🔴 Agent B 답:     Hello! I don't have hobbies, but I enjoy helping and chatting with you. What's on your mind today?

── Round 2 ──────────────────────────────────────────────────────
  → Agent A 가 받음:  "Hello! I don't have hobbies, but I enjoy helping and chatting with you. What's on your mind today?"
  🔵 Agent A 답:     Hello! I'm glad to hear that you enjoy helping and chatting. Today, my focus is on assisting you. How can I support you right now?
  → Agent B 가 받음:  "Hello! I'm glad to hear that you e

## 6. LangSmith 분산 트레이싱

Agent Server 는 A2A 의 `contextId` 를 LangSmith 의 `thread_id` 로 **자동 매핑**합니다. 덕분에 여러 에이전트를 오가는 대화가 추가 설정 없이 하나의 thread 로 묶입니다.

여러 에이전트의 트레이스를 **하나로 통합**하려면:
1. 후속 턴 메시지에 `contextId` / `taskId` 포함
2. JSON-RPC 최상위 `metadata` 에 `thread_id` 전달
3. **모든 에이전트에서 같은 `thread_id` 재사용**

In [81]:
async def run_conversation():
    """§5 와 같지만, 두 에이전트가 동일한 thread_id 를 공유한다.
    LangSmith 에서 두 에이전트의 호출이 하나의 thread 트레이스로 묶여 보인다."""
    agent_a_id = os.getenv("AGENT_A_ID")
    agent_b_id = os.getenv("AGENT_B_ID")
    if not agent_a_id or not agent_b_id:
        print("⚠️ .env 의 AGENT_A_ID / AGENT_B_ID 가 비어 있습니다.")
        print("   채우고 노트북 상단의 load_dotenv 셀을 다시 실행하거나 커널을 재시작하세요.")
        return

    # 두 에이전트가 공유할 LangSmith thread_id (대화 전체의 고유 id)
    thread_id = str(uuid.uuid4())
    seed = "Hello! Let's collaborate."

    print(f"🧵 LangSmith thread_id (공유): {thread_id}")
    print(f"   🔵 Agent A  port 2024  assistant_id={agent_a_id}")
    print(f"   🔴 Agent B  port 2025  assistant_id={agent_b_id}")
    print(f"💬 Seed (사용자 → Agent A):  {seed}")
    print("=" * 66, flush=True)

    # contextId · taskId 는 서버 단위 → 에이전트별로 따로 유지
    ctx_a, task_a = None, None
    ctx_b, task_b = None, None
    message = seed

    async with aiohttp.ClientSession() as session:
        for i in range(1, 4):
            print(f"\n── Round {i} " + "─" * 54, flush=True)

            print(f"  → Agent A 가 받음:  {message!r}", flush=True)
            reply_a, ctx_a, task_a = await send_message(
                session, 2024, agent_a_id, message,
                context_id=ctx_a, task_id=task_a,
                thread_id=thread_id,   # ★ 공유 thread_id
            )
            print(f"  🔵 Agent A 답:     {reply_a}", flush=True)

            print(f"  → Agent B 가 받음:  {reply_a!r}", flush=True)
            reply_b, ctx_b, task_b = await send_message(
                session, 2025, agent_b_id, reply_a,
                context_id=ctx_b, task_id=task_b,
                thread_id=thread_id,   # ★ 공유 thread_id
            )
            print(f"  🔴 Agent B 답:     {reply_b}", flush=True)

            message = reply_b

    print("\n" + "=" * 66)
    print(f"✅ 대화 종료 — LangSmith 에서 thread_id={thread_id} 로 통합 트레이스 확인")
    print(f"   (LangSmith → Threads → {thread_id})", flush=True)


# 주피터에서는 await 로 실행 (스크립트면 asyncio.run(run_conversation()))
await run_conversation()

🧵 LangSmith thread_id (공유): 0410b265-ef97-4ab1-b88c-0922aa30f22f
   🔵 Agent A  port 2024  assistant_id=fe096781-5601-53d2-b2f6-0d3403f7e9ca
   🔴 Agent B  port 2025  assistant_id=fe096781-5601-53d2-b2f6-0d3403f7e9ca
💬 Seed (사용자 → Agent A):  Hello! Let's collaborate.

── Round 1 ──────────────────────────────────────────────────────
  → Agent A 가 받음:  "Hello! Let's collaborate."
  🔵 Agent A 답:     Hello! I'd be happy to collaborate with you. What do you have in mind?
  → Agent B 가 받음:  "Hello! I'd be happy to collaborate with you. What do you have in mind?"
  🔴 Agent B 답:     Hello! I'm here to assist you with any questions or tasks you have. Just let me know how I can help!

── Round 2 ──────────────────────────────────────────────────────
  → Agent A 가 받음:  "Hello! I'm here to assist you with any questions or tasks you have. Just let me know how I can help!"
  🔵 Agent A 답:     Hello! I'm here to help with any questions or tasks you have. Just let me know what you need assistance with!


## 7. 비-LangGraph 에이전트 트레이싱 & A2A 끄기

LangGraph 가 아닌 에이전트는 들어오는 A2A `metadata` 에서 `thread_id` 를 직접 꺼내 OTel span 에 붙입니다. 또 필요하면 `langgraph.json` 에서 A2A 를 끌 수 있습니다.

In [85]:
# §7. 비-LangGraph(FastAPI) 에이전트에서 thread_id 추출 → LangSmith 트레이싱
#
# 0) 의존성 설치 — fastapi 와 LangSmith OTel 통합 (한 번만; 이미 있으면 빠르게 통과)
get_ipython().system('uv pip install -q fastapi "langsmith[otel]"')

# 1) 환경변수 적용 — .env 의 LangSmith 키를 로드하고 검증
import os
# from dotenv import load_dotenv

# load_dotenv("/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/.env", override=True)

required = ["LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    print(f"⚠️ 다음 환경변수가 비어 있습니다: {missing}")
    print("   .env 에 채우고 이 셀을 다시 실행하세요.")

# LangSmith 트레이싱 활성화 (이미 .env 에 있으면 그 값을 유지)
os.environ.setdefault("LANGSMITH_TRACING", "true")

print(f"LANGSMITH_PROJECT  = {os.environ.get('LANGSMITH_PROJECT')!r}")
print(f"LANGSMITH_TRACING  = {os.environ.get('LANGSMITH_TRACING')!r}")
print(f"LANGSMITH_API_KEY  = {'(set)' if os.environ.get('LANGSMITH_API_KEY') else '(missing)'}")

# 2) FastAPI 미들웨어 + LangSmith OTel 구성
import json
from fastapi import FastAPI, Request
from langsmith.integrations.otel import configure as configure_otel
from opentelemetry import trace

# project_name 은 .env 의 LANGSMITH_PROJECT 와 같게 (없으면 기본값)
configure_otel(project_name=os.environ.get("LANGSMITH_PROJECT", "week3-protocols"))
tracer = trace.get_tracer(__name__)

app = FastAPI()


@app.middleware("http")
async def set_thread_id_middleware(request: Request, call_next):
    """들어오는 A2A 요청의 metadata.thread_id 를 꺼내 OTel span 속성으로 붙인다.
    이렇게 하면 LangSmith 가 같은 thread_id 의 호출을 하나의 thread 트레이스로 묶는다."""
    thread_id = None
    if request.method == "POST":
        body_bytes = await request.body()
        if body_bytes:
            try:
                body = json.loads(body_bytes)
                thread_id = body.get("metadata", {}).get("thread_id")
            except Exception:
                pass
            # body 를 한 번 읽었으니, downstream 핸들러가 다시 읽을 수 있게 receive 를 재설정
            async def receive():
                return {"type": "http.request", "body": body_bytes}
            request._receive = receive

    with tracer.start_as_current_span("agent") as span:
        if thread_id:
            span.set_attribute("langsmith.metadata.thread_id", thread_id)
        return await call_next(request)


print("\n✅ FastAPI app + LangSmith OTel 미들웨어 준비 완료")
print("   (이 셀은 패턴 데모입니다 — 실제 서비스는 별도 터미널에서 uvicorn 으로 기동)")

LANGSMITH_PROJECT  = 'week3-protocols'
LANGSMITH_TRACING  = 'true'
LANGSMITH_API_KEY  = (set)

✅ FastAPI app + LangSmith OTel 미들웨어 준비 완료
   (이 셀은 패턴 데모입니다 — 실제 서비스는 별도 터미널에서 uvicorn 으로 기동)


In [86]:
# langgraph.json 에서 A2A 비활성화
disable_a2a = '''
{
  "$schema": "https://langgra.ph/schema.json",
  "http": {
    "disable_a2a": true
  }
}
'''
print(disable_a2a)


{
  "$schema": "https://langgra.ph/schema.json",
  "http": {
    "disable_a2a": true
  }
}



## 정리 & 연습 문제

**핵심 요약**
- A2A = **에이전트 ↔ 에이전트** 표준 메시징. 엔드포인트 `POST /a2a/{assistant_id}` (JSON-RPC 2.0)
- 메서드: `message/send`, `message/stream`, `tasks/get` / 디스커버리: `agent-card.json`
- 에이전트 state 에 **`messages` 키 필수**
- `contextId`(스레드) + `taskId`(요청)로 대화 이어감 — 첫 턴엔 생략, 이후 포함
- `contextId → thread_id` 자동 매핑 + `metadata.thread_id` 로 멀티 에이전트 트레이스 통합

**연습**
1. `graph.py` 의 system prompt 를 서로 다른 페르소나(낙관론자 vs 비관론자)로 둔 두 에이전트를 띄우고 핑퐁 대화를 돌려 보세요.
2. 같은 `thread_id` 로 호출한 뒤 LangSmith 에서 두 에이전트 트레이스가 한 thread 로 묶이는지 확인해 보세요.
3. `message/stream` 으로 바꿔 SSE 스트리밍 응답을 받아 보세요.

**참고 레포**
- 두 LangGraph 에이전트: https://github.com/langchain-samples/A2A-langgraph
- Google ADK + LangChain: https://github.com/langchain-samples/A2A-google-adk
- A2A 공식 명세: https://a2a-protocol.org/latest/